# DIL Results Analysis
Use the helper functions in `data_analysis_fcns.DIL_Metrics` to visualize confusion matrices, imbalanced-learning metrics, and population statistics across multiple runs.

In [1]:
# Imports and helper fcns
from data_analysis_fcns.DIL_Metrics import (
    plot_confusion_matrices,
    plot_imbalanced_metrics,
    plot_population_statistics,
)

from pathlib import Path
import json
import re

def read_jsonc(path: str):
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()
    raw = re.sub(r"//.*", "", raw)
    raw = re.sub(r"/\*.*?\*/", "", raw, flags=re.DOTALL)
    return json.loads(raw)


In [2]:
# --- Configure which configs to compare (use config names without .jsonc)

metaconfig = "meta_configs/core50_DIL_retry.jsonc" # If metaconfig is used, configs is ignored
configs = []
metapath = Path(metaconfig)
if metapath.is_file():
    metadata = read_jsonc(metaconfig)
    configs = metadata.get("configs")
    configs = [configname.replace('configs/', '') for configname in configs]
    configs = [configname.replace('.jsonc', '') for configname in configs]
    
else:
    configs = ["mini_debug", "example_config"]  # adjust to your setup

labels = None  # optional display labels for the configs
# Example single result file for quick inspection
result_file = 'results/mini_debug_vit_imagelevel/vit_moe_imagelevel_cifar10_04241501.pt'  # change to one of your saved .pt files

## Confusion matrices
Plot per-domain confusion matrices (model vs baseline) and/or epoch-by-epoch full-test confusion matrices. If you leave all options unset the function will plot all available confusion data.

In [ ]:
# Plot domain-level comparisons and the epoch list (toggle arguments as desired)
plot_confusion_matrices(result_file, per_domain=True, compare_baseline=True, per_epoch=False)

## Imbalanced-learning metrics per epoch
Compute macro-averaged recall/precision/F1 plus MAUC and G-Mean per epoch from the epoch-level confusion matrices.

In [ ]:
# Plot imbalanced-learning metrics for the chosen result file
plot_imbalanced_metrics(result_file)

## Population statistics across configs
Aggregate multiple runs per config (searching under the `results/` tree for filenames or folders that contain each config name) and plot mean +/- std shading for each metric.

In [ ]:
# Compare configured experiments using saved .pt files under results/

# add save_directory_name to save the plots to a specific directory as svgs
plot_population_statistics(configs, results_root='results', labels=labels), #save_directory_name='OH_full')

## Expert usage analysis
Plot per-layer expert utilization stored in the results file (if available).

In [ ]:
# Load saved results and plot expert usage per layer
import torch
from pathlib import Path
import matplotlib.pyplot as plt

def plot_expert_usage(result_path):
    data = torch.load(result_path, weights_only=False)
    cum = data.get('expert_cumulative_usage', None)
    per_epoch = data.get('expert_usage_history', None)
    if cum is None and per_epoch is None:
        print('No expert usage data found in', result_path)
        return
    # plot cumulative usage if present (list of lists per layer)
    if cum is not None:
        for li, layer in enumerate(cum):
            plt.figure()
            plt.bar(range(len(layer)), layer)
            plt.title(f'Layer {li} cumulative image counts per expert')
            plt.xlabel('expert')
            plt.ylabel('image count')
            plt.show()
    # plot last epoch per-layer usage (if history exists)
    if per_epoch is not None and len(per_epoch) > 0:
        last = per_epoch[-1]
        # last expected to be a list of per-layer lists (or None)
        for li, layer in enumerate(last):
            try:
                plt.figure()
                plt.bar(range(len(layer)), layer)
                plt.title(f'Layer {li} epoch image counts per expert (last)')
                plt.xlabel('expert')
                plt.ylabel('image count')
                plt.show()
            except Exception:
                pass

# call with the chosen result file
plot_expert_usage(result_file)

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

### Expert usage histograms after each domain
The following cell loads the saved results file and plots per-layer expert usage histograms taken at the epoch where each domain finished.
If confusion matrices are available in the saved file, an approximate per-class per-expert heatmap is also shown (approximation distributes expert counts proportionally to class frequencies).

In [ ]:
# Plot per-layer expert usage (one plot per MoE layer).
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import os

# load results (weights_only=False for compatibility with older save format)
data = torch.load(result_file, weights_only=False)
expert_hist = data.get('expert_usage_history', None)
domain_boundaries = data.get('domain_boundaries', None)
confusions = data.get('confusion_matrices', None)

if expert_hist is None:
    print('No expert usage history found in', result_file)
else:
    # determine number of domains and epoch indices for domain ends
    if domain_boundaries is None:
        print('Warning: domain_boundaries missing in saved results — falling back to last-available epoch per domain')
        if confusions is not None and len(confusions) > 0:
            num_domains = len(confusions)
        else:
            num_domains = None
            for snap in expert_hist:
                if snap is None:
                    continue
                try:
                    num_domains = len(snap)
                    break
                except Exception:
                    continue
            if num_domains is None:
                print('Could not infer number of domains from saved data; aborting')
                num_domains = 0
        # compute per-domain epoch index as last epoch where snapshot for that domain exists
        domain_epoch_indices = []
        for d in range(num_domains):
            found = False
            for epoch_idx in range(len(expert_hist) - 1, -1, -1):
                snap = expert_hist[epoch_idx]
                if snap is None:
                    continue
                if d < len(snap) and snap[d] is not None:
                    domain_epoch_indices.append(epoch_idx)
                    found = True
                    break
            if not found:
                domain_epoch_indices.append(None)
    else:
        num_domains = len(domain_boundaries)
        domain_epoch_indices = [max(0, b - 1) for b in domain_boundaries]

    if num_domains == 0:
        print('No domains to plot.')
    else:
        # find dims
        num_layers = None
        num_experts = None
        for epoch_idx in range(len(expert_hist)):
            snap = expert_hist[epoch_idx]
            if snap is None:
                continue
            # find first domain with data
            for d in range(len(snap)):
                dom = snap[d]
                if dom is None:
                    continue
                num_layers = len(dom)
                num_experts = int(len(dom[0])) if num_layers > 0 else 0
                break
            if num_layers is not None:
                break

        if num_layers is None or num_experts is None or num_layers == 0 or num_experts == 0:
            print('Could not determine layers/experts from saved snapshots; aborting plotting.')
        else:
            counts_per_layer = [np.zeros((num_domains, num_experts), dtype=float) for _ in range(num_layers)]
            for d in range(num_domains):
                epoch_idx = domain_epoch_indices[d]
                if epoch_idx is None:
                    continue
                snap = expert_hist[epoch_idx]
                if snap is None or d >= len(snap) or snap[d] is None:
                    continue
                domain_usage = snap[d]
                for li in range(min(num_layers, len(domain_usage))):
                    vals = domain_usage[li] if domain_usage[li] is not None else []
                    vals = np.array(vals, dtype=float) if len(vals) > 0 else np.zeros(num_experts)
                    counts_per_layer[li][d, : vals.size] = vals

            for li in range(num_layers):
                arr = counts_per_layer[li]
                # Normalize rows to sum to 1 (percentages)
                row_sums = arr.sum(axis=1, keepdims=True)
                # Prevent division by zero
                row_sums[row_sums == 0] = 1 
                arr_normalized = arr / row_sums
                fig, ax = plt.subplots(figsize=(max(6, num_domains * 0.7), 4))
                x = np.arange(num_domains)
                width = 0.8 / max(1, num_experts)
                for e in range(num_experts):
                    ax.bar(x + e * width, arr_normalized[:, e], width=width, label=f'expert {e}')
                ax.set_xticks(x)
                ax.set_xticklabels([f'D{d}' for d in range(num_domains)], rotation=45)
                ax.set_title(f'Layer {li} expert usage per domain (snapshot at domain end)')
                ax.set_xlabel('domain')
                ax.set_ylabel('image count')
                ax.legend()
                plt.tight_layout()
                plt.show()
                
                # uncomment the below to save as svg
                # out_path = os.path.join('images', f"{os.path.basename(result_file)}_expert{li}.svg")
                # fig.savefig(out_path, format="svg", bbox_inches="tight")




UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 